# E-commerce Sales & Customer Insights Dashboard

**Prepared by:** C Chetan &nbsp;|&nbsp; **Project:** Data Analytics Internship  
**Dataset:** UCI Online Retail II, a UK online gift retailer (Dec 2009 – Dec 2011, about 1.07 million order lines)  
**Source:** https://archive.ics.uci.edu/dataset/502/online+retail+ii

**Objective:** identify the main revenue drivers and business opportunities by analysing sales, customers, products and geography.

### Project workflow
1. Load and combine the two yearly sheets.
2. Understand and clean the transaction data.
3. Calculate sales and customer KPIs.
4. Analyse monthly trends and customer cohorts.
5. Segment customers using simple RFM analysis.
6. Analyse products, categories and geography.
7. Produce data-backed business recommendations.
8. Use the separate Streamlit app for an interactive dashboard.

**Before running:** upload `online_retail_II.xlsx` to the Colab Files panel. If it is not found, the loading cell also gives you an upload option; a download fallback is provided for environments where Colab upload is unavailable. Then choose **Runtime → Run all**.

## 1. Setup and Data Loading

In [ ]:
# Setup
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 50)

In [ ]:
# Get the dataset
# In Google Colab, upload the Excel file if it is not already available.
# This keeps the notebook easy to submit: the dataset is NOT embedded inside the notebook.

import os

DATA_FILE = "online_retail_II.xlsx"

if not os.path.exists(DATA_FILE):
    try:
        # Google Colab upload box
        from google.colab import files
        print("Dataset not found. Please upload online_retail_II.xlsx")
        uploaded = files.upload()
        if DATA_FILE not in uploaded:
            # Use the first uploaded Excel file if its name is different.
            excel_files = [name for name in uploaded if name.lower().endswith((".xlsx", ".xls"))]
            if excel_files:
                os.rename(excel_files[0], DATA_FILE)
    except ImportError:
        # Fallback for Jupyter/local environments without Google Colab.
        print("Colab upload is unavailable. Trying the UCI download instead...")
        os.system('wget -q -O online_retail_ii.zip "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"')
        os.system('unzip -o -q online_retail_ii.zip')

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        "online_retail_II.xlsx was not found. Upload the dataset and run this cell again."
    )

print("Dataset ready:", os.path.exists(DATA_FILE))

Dataset ready: True


In [ ]:
# Load both yearly sheets and combine them (takes 1-3 minutes)
sheets = pd.read_excel("online_retail_II.xlsx", sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)
print("Sheets:", list(sheets.keys()))
print("Combined shape:", df.shape)
df.head()

Sheets: ['Year 2009-2010', 'Year 2010-2011']
Combined shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


## 2. Data Understanding

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isna().sum())
print("\nQuantity / Price summary:")
df[['Quantity','Price']].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB

Missing values per column:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Quantity / Price summary:


,Quantity,Price
count,"1,067,371.00","1,067,371.00"
mean,9.94,4.65
std,172.71,123.55
min,"-80,995.00","-53,594.36"
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,"80,995.00","38,970.00"


*Insight: about 23% of rows have no Customer ID (guest orders), and the data contains negative quantities (returns) and zero or negative prices that need cleaning.*

## 3. Data Cleaning

The cleaning steps are deliberately simple and visible so they can be explained in an internship review:

- Remove exact duplicate rows.
- Separate refund/cancellation invoices (invoice numbers beginning with `C`).
- Remove rows with zero/negative quantity or price.
- Remove non-product stock codes such as postage and fees.
- Remove rows without a product description.
- Standardise product text and country names.
- Keep missing Customer IDs as **guest orders** rather than inventing customer information.

In [ ]:
# Cleaning
df = df.rename(columns={
    'Invoice':'invoice', 'StockCode':'stock_code', 'Description':'description',
    'Quantity':'quantity', 'InvoiceDate':'order_date', 'Price':'unit_price',
    'Customer ID':'customer_id', 'Country':'country'})

log = {"Raw rows": len(df)}

# a) duplicates
df = df.drop_duplicates()
log["After removing duplicates"] = len(df)

# b) refunds: invoices that start with 'C'
df['invoice'] = df['invoice'].astype(str)
is_refund = df['invoice'].str.startswith('C')
refunds = df[is_refund].copy()
df = df[~is_refund]
log["After removing refund invoices"] = len(df)

# c) bad quantity / price
df = df[(df['quantity'] > 0) & (df['unit_price'] > 0)]
log["After removing zero/negative qty or price"] = len(df)

# d) non-product stock codes
df['stock_code'] = df['stock_code'].astype(str).str.upper()
non_product = ['POST','D','M','DOT','BANK CHARGES','CRUK','AMAZONFEE','S','B','ADJUST','TEST001','TEST002']
df = df[~df['stock_code'].isin(non_product)]
log["After removing non-product codes"] = len(df)

# e) missing descriptions, tidy text and country names
df = df.dropna(subset=['description'])
df['description'] = df['description'].str.strip().str.upper()
df['country'] = df['country'].replace({'EIRE':'Ireland', 'RSA':'South Africa',
                                       'USA':'United States', 'Unspecified':'Unknown'})
log["Final clean rows"] = len(df)

# new columns we need later
df['revenue'] = df['quantity'] * df['unit_price']
df['order_month'] = df['order_date'].dt.to_period('M').dt.to_timestamp()

pd.DataFrame({"rows": log})

,rows
Raw rows,1067371
After removing duplicates,1033036
After removing refund invoices,1013932
After removing zero/negative qty or price,1007913
After removing non-product codes,1003704
Final clean rows,1003704


*Insight: 63,667 rows (6%) were removed, leaving 1,003,704 clean order lines.*

**Product categories:** the dataset has no category column, so categories are derived from keywords in product names (an approximation). It also has no cost data, so margin cannot be calculated and revenue is used instead.

In [ ]:
# Keyword-based categories (first match wins, so order matters)
category_keywords = {
    'Christmas & Seasonal':     ['CHRISTMAS','XMAS','EASTER','ADVENT','SANTA'],
    'Lighting & Candles':       ['CANDLE','LIGHT','LAMP','LANTERN'],
    'Kitchen & Dining':         ['MUG','CUP','PLATE','BOWL','TEAPOT','CAKE','SPOON','JUG','GLASS','BOTTLE','KITCHEN'],
    'Bags & Storage':           ['BAG','BOX','TIN','BASKET','JAR','CASE'],
    'Cards, Paper & Gift Wrap': ['CARD','WRAP','PAPER','RIBBON','NOTEBOOK','STICKER','GIFT'],
    'Home Decor':               ['FRAME','CUSHION','CLOCK','SIGN','MIRROR','HOOK','DOORMAT','HANGING','HEART','WALL','VASE'],
    'Toys & Games':             ['TOY','GAME','DOLL','PUZZLE','PLAYHOUSE','SKIPPING'],
}

df['category'] = 'Other'
for cat, words in reversed(list(category_keywords.items())):
    pattern = '|'.join(words)
    df.loc[df['description'].str.contains(pattern, regex=True), 'category'] = cat

df['category'].value_counts()

,count
category,
Other,277571
Bags & Storage,191008
Home Decor,164577
Kitchen & Dining,154771
Lighting & Candles,74911
"Cards, Paper & Gift Wrap",74764
Christmas & Seasonal,49583
Toys & Games,16519


## 4. Key Performance Indicators

In [ ]:
# KPIs
df_c = df.dropna(subset=['customer_id']).copy()
df_c['customer_id'] = df_c['customer_id'].astype(int)

total_revenue   = df['revenue'].sum()
total_orders    = df['invoice'].nunique()
total_customers = df_c['customer_id'].nunique()
aov             = total_revenue / total_orders

orders_per_customer = df_c.groupby('customer_id')['invoice'].nunique()
repeat_rate = (orders_per_customer > 1).mean()

refund_value = (refunds['quantity'].abs() * refunds['unit_price']).sum()
refund_rate  = refund_value / total_revenue
guest_share  = 1 - len(df_c) / len(df)

print(f"Total revenue            : £{total_revenue:,.0f}")
print(f"Total orders             : {total_orders:,}")
print(f"Identified customers     : {total_customers:,}")
print(f"Average order value (AOV): £{aov:,.2f}")
print(f"Repeat-customer rate     : {repeat_rate:.1%}")
print(f"Refund value / revenue   : {refund_rate:.1%}")
print(f"Rows from guest orders   : {guest_share:.1%}")

Total revenue            : £19,659,775
Total orders             : 39,526
Identified customers     : 5,853
Average order value (AOV): £497.39
Repeat-customer rate     : 72.3%
Refund value / revenue   : 7.4%
Rows from guest orders   : 22.6%


*Insight: £19.7M revenue from 39,526 orders. The high average order value reflects many wholesale buyers. 72% of customers order more than once, and refunds equal 7.4% of revenue.*

### Combined data model
The raw file is one flat table, so separate customer, product and location tables are built from it and joined back to the order lines.

In [ ]:
# Build customer, product and location tables from the cleaned orders
customers = (df_c.groupby('customer_id')
               .agg(country=('country', 'last'), first_order=('order_date', 'min'),
                    orders=('invoice', 'nunique'), revenue=('revenue', 'sum'))
               .reset_index())

products = (df.groupby('stock_code')
              .agg(description=('description', 'first'), category=('category', 'first'),
                   units_sold=('quantity', 'sum'), revenue=('revenue', 'sum'))
              .reset_index())

locations = (df.groupby('country')
               .agg(orders=('invoice', 'nunique'), customers=('customer_id', 'nunique'),
                    revenue=('revenue', 'sum'))
               .reset_index())

# Join the order lines to the customer and product tables
orders_full = (df[['invoice', 'order_date', 'stock_code', 'customer_id', 'country', 'quantity', 'revenue']]
               .merge(products[['stock_code', 'description', 'category']], on='stock_code', how='left')
               .merge(customers[['customer_id', 'first_order']], on='customer_id', how='left'))

orders_full['customer_id'] = orders_full['customer_id'].astype('Int64')   # show IDs as whole numbers

print(f"Orders: {len(df):,} rows | Customers: {len(customers):,} | "
      f"Products: {len(products):,} | Locations: {len(locations):,}")
print(f"Combined orders table: {orders_full.shape[0]:,} rows x {orders_full.shape[1]} columns "
      f"(row count unchanged: {len(orders_full) == len(df)})")
orders_full.head()

Orders: 1,003,704 rows | Customers: 5,853 | Products: 4,735 | Locations: 43
Combined orders table: 1,003,704 rows x 10 columns (row count unchanged: True)


,invoice,order_date,stock_code,customer_id,country,quantity,revenue,description,category,first_order
0,489434,2009-12-01 07:45:00,85048,13085,United Kingdom,12,83.40,15CM CHRISTMAS GLASS BALL 20 LIGHTS,Christmas & Seasonal,2009-12-01 07:45:00
1,489434,2009-12-01 07:45:00,79323P,13085,United Kingdom,12,81.00,PINK CHERRY LIGHTS,Lighting & Candles,2009-12-01 07:45:00
2,489434,2009-12-01 07:45:00,79323W,13085,United Kingdom,12,81.00,WHITE CHERRY LIGHTS,Lighting & Candles,2009-12-01 07:45:00
3,489434,2009-12-01 07:45:00,22041,13085,United Kingdom,48,100.80,"RECORD FRAME 7"" SINGLE SIZE",Home Decor,2009-12-01 07:45:00
4,489434,2009-12-01 07:45:00,21232,13085,United Kingdom,24,30.00,STRAWBERRY CERAMIC TRINKET BOX,Bags & Storage,2009-12-01 07:45:00


*Insight: 1,003,704 order lines link to 5,853 customers, 4,735 products and 43 countries; the join keeps every order line.*

## 5. Monthly Trend

In [ ]:
# Revenue and orders per month
monthly = df.groupby('order_month').agg(revenue=('revenue','sum'),
                                        orders=('invoice','nunique')).reset_index()

fig = px.bar(monthly, x='order_month', y='revenue', title='Monthly Revenue (£)',
             labels={'order_month':'Month','revenue':'Revenue (£)'})
fig.show()

fig = px.line(monthly, x='order_month', y='orders', markers=True, title='Monthly Orders',
              labels={'order_month':'Month','orders':'Orders'})
fig.show()

*Insight: revenue peaks every November (£1.43M in 2010, £1.45M in 2011) and dips in January–February. Dec 2011 is incomplete because the data ends on 9 December.*

## 6. Cohort Retention

In [ ]:
# Cohort retention heatmap
df_c['cohort_month'] = df_c.groupby('customer_id')['order_month'].transform('min')
df_c['month_index'] = ((df_c['order_month'].dt.year  - df_c['cohort_month'].dt.year) * 12 +
                       (df_c['order_month'].dt.month - df_c['cohort_month'].dt.month))

cohort_counts = (df_c.groupby(['cohort_month','month_index'])['customer_id']
                     .nunique().reset_index())
pivot = cohort_counts.pivot(index='cohort_month', columns='month_index', values='customer_id')
retention = pivot.div(pivot[0], axis=0) * 100
retention.index = retention.index.strftime('%Y-%m')

fig = px.imshow(retention.iloc[:, :13], text_auto='.0f', aspect='auto',
                color_continuous_scale='Blues',
                labels=dict(x='Months since first purchase', y='Cohort', color='% active'),
                title='Customer Retention by Cohort (%)')
fig.show()
print(f"Average month-1 retention: {retention[1].mean():.1f}%")

Average month-1 retention: 21.0%


*Insight: only about 21% of new customers buy again in their second month, so early retention is a weak point.*

## 7. Customer Segmentation (RFM)

Customers are scored 1–4 on three simple measures:

- **Recency:** how recently the customer purchased.
- **Frequency:** how many different orders the customer made.
- **Monetary:** how much revenue the customer generated.

The three scores are combined into four easy-to-explain segments: **Champions, At Risk, New / Promising, and Hibernating**.

In [ ]:
# RFM table
snapshot = df_c['order_date'].max() + pd.Timedelta(days=1)

rfm = df_c.groupby('customer_id').agg(
    recency   = ('order_date', lambda x: (snapshot - x.max()).days),
    frequency = ('invoice', 'nunique'),
    monetary  = ('revenue', 'sum'))

rfm['R'] = pd.qcut(rfm['recency'], 4, labels=[4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'], 4, labels=[1,2,3,4]).astype(int)

def segment(row):
    if row['R'] >= 3 and row['F'] >= 3: return 'Champions'
    if row['R'] <= 2 and row['F'] >= 3: return 'At Risk (valuable, slipping)'
    if row['R'] >= 3 and row['F'] <= 2: return 'New / Promising'
    return 'Hibernating'

rfm['segment'] = rfm.apply(segment, axis=1)

seg = rfm.groupby('segment').agg(customers=('monetary','size'),
                                 revenue=('monetary','sum'),
                                 avg_orders=('frequency','mean'),
                                 avg_recency_days=('recency','mean')).reset_index()
seg['revenue_share_%'] = seg['revenue'] / seg['revenue'].sum() * 100
display(seg.sort_values('revenue', ascending=False))

fig = px.bar(seg, x='segment', y='revenue', color='segment',
             title='Revenue by Customer Segment (£)')
fig.update_layout(showlegend=False)
fig.show()

,segment,customers,revenue,avg_orders,avg_recency_days,revenue_share_%
1,Champions,2040,"13,096,727.07",12.86,29.57,76.67
0,"At Risk (valuable, slipping)",886,"2,119,761.84",6.51,275.89,12.41
2,Hibernating,2037,"1,118,116.65",1.48,408.28,6.55
3,New / Promising,890,"747,254.22",1.77,40.04,4.37


*Insight: Champions are 35% of customers (2,040) but generate 77% of revenue. 886 At-Risk customers represent £2.1M in past revenue and have been inactive for about 276 days on average.*

## 8. Top Products and Category Performance

In [ ]:
# Top products
top_products = (df.groupby('description')['revenue'].sum()
                  .sort_values(ascending=False).head(10).reset_index())
fig = px.bar(top_products.sort_values('revenue'), x='revenue', y='description', orientation='h',
             title='Top 10 Products by Revenue (£)')
fig.show()

In [ ]:
# Categories: which ones underperform?
#     Revenue per product (SKU) shows if a category has many products but weak sales
cat = df.groupby('category').agg(revenue=('revenue','sum'),
                                 units=('quantity','sum'),
                                 products=('stock_code','nunique')).reset_index()
cat['revenue_per_product'] = cat['revenue'] / cat['products']
cat['share_%'] = cat['revenue'] / cat['revenue'].sum() * 100
display(cat.sort_values('revenue_per_product'))

fig = px.bar(cat.sort_values('revenue_per_product'), x='revenue_per_product', y='category',
             orientation='h', title='Revenue per Product by Category (£)  - lowest = underperforming')
fig.show()

,category,revenue,units,products,revenue_per_product,share_%
6,Other,"4,841,316.76",3008402,1779,"2,721.37",24.63
1,"Cards, Paper & Gift Wrap","1,128,716.20",1067189,339,"3,329.55",5.74
2,Christmas & Seasonal,"934,858.02",614019,260,"3,595.61",4.76
5,Lighting & Candles,"1,783,683.87",1028162,489,"3,647.62",9.07
7,Toys & Games,"267,503.12",121402,66,"4,053.08",1.36
3,Home Decor,"3,319,557.37",1513897,735,"4,516.40",16.89
4,Kitchen & Dining,"3,179,503.07",1705763,613,"5,186.79",16.17
0,Bags & Storage,"4,204,636.45",2129578,516,"8,148.52",21.39


*Insight: Regency Cakestand 3 Tier is the top product (£331k). Bags & Storage earns the most per product (£8.1k) and Cards, Paper & Gift Wrap the least among named categories (£3.3k). "Other" is a catch-all group and is not treated as a category.*

## 9. High-Value Customers

In [ ]:
# High-value customers
cust_rev = df_c.groupby('customer_id')['revenue'].sum().sort_values(ascending=False)
top10_pct_share = cust_rev.head(int(len(cust_rev)*0.10)).sum() / cust_rev.sum()
print(f"Top 10% of customers generate {top10_pct_share:.1%} of revenue\n")

top_customers = rfm.sort_values('monetary', ascending=False).head(10)[
    ['monetary','frequency','recency','segment']]
top_customers

Top 10% of customers generate 63.9% of revenue



,monetary,frequency,recency,segment
customer_id,,,,
18102,"580,987.04",145,1,Champions
14646,"526,751.52",145,2,Champions
14156,"304,719.88",145,10,Champions
14911,"279,477.79",375,1,Champions
17450,"244,784.25",51,8,Champions
13694,"195,640.69",143,4,Champions
17511,"172,132.87",60,3,Champions
16446,"168,472.50",2,1,New / Promising
16684,"147,142.77",55,4,Champions


*Insight: the top 10% of customers generate 64% of revenue; the largest single customer (ID 18102) spent about £581k.*

## 10. Regional Patterns

In [ ]:
# Geography
country = df.groupby('country').agg(revenue=('revenue','sum'),
                                    orders=('invoice','nunique')).reset_index()
country['aov'] = country['revenue'] / country['orders']
uk_share = country.loc[country['country']=='United Kingdom','revenue'].sum() / country['revenue'].sum()
print(f"United Kingdom share of revenue: {uk_share:.1%}")

non_uk = country[country['country']!='United Kingdom'].sort_values('revenue', ascending=False).head(10)
fig = px.bar(non_uk.sort_values('revenue'), x='revenue', y='country', orientation='h',
             title='Top 10 Export Markets by Revenue (£, excluding UK)')
fig.show()

# countries with enough orders to compare fairly
big = country[country['orders'] >= 20].sort_values('aov', ascending=False).head(10)
fig = px.bar(big, x='country', y='aov', title='Highest Average Order Value by Country (min 20 orders)')
fig.show()

United Kingdom share of revenue: 85.5%


*Insight: the UK accounts for 85.5% of revenue. The largest export markets are Ireland (£633k), the Netherlands (£550k) and Germany (£383k). The Netherlands has the highest average order value (£2,545).*

## 11. Data-Backed Recommendations

In [ ]:
# Recommendations built from the numbers above
champ = seg[seg['segment']=='Champions'].iloc[0]
risk  = seg[seg['segment'].str.startswith('At Risk')].iloc[0]
named_cats = cat[cat['category'] != 'Other']          # ignore the catch-all bucket
worst_cat = named_cats.sort_values('revenue_per_product').iloc[0]
best_cat  = named_cats.sort_values('revenue', ascending=False).iloc[0]
q4_share  = (monthly[monthly['order_month'].dt.month.isin([9,10,11])]['revenue'].sum()
             / monthly['revenue'].sum())
top_export = non_uk.iloc[0]

recs = [
 f"1. PROTECT CHAMPIONS: {int(champ['customers']):,} customers produce {champ['revenue_share_%']:.0f}% of revenue. "
   f"Launch a loyalty / trade-account programme with early access to new stock.",
 f"2. WIN BACK AT-RISK CUSTOMERS: {int(risk['customers']):,} previously frequent buyers have gone quiet "
   f"(avg {risk['avg_recency_days']:.0f} days since last order) and represent £{risk['revenue']:,.0f} of past revenue. "
   f"Send a targeted win-back offer.",
 f"3. FIX RETENTION: only ~{retention[1].mean():.0f}% of new customers return the next month. "
   f"Add a follow-up email/discount for the first 30 days after a first order.",
 f"4. PLAN FOR PEAK SEASON: Sep-Nov brings {q4_share:.0%} of the revenue in the data. Secure stock and "
   f"marketing budget before September.",
 f"5. REVIEW WEAK CATEGORY: '{worst_cat['category']}' earns the least per product "
   f"(£{worst_cat['revenue_per_product']:,.0f}). Cut slow SKUs or bundle them with '{best_cat['category']}' best-sellers.",
 f"6. GROW EXPORTS: the UK is {uk_share:.0%} of revenue. {top_export['country']} is the largest export market "
   f"(£{top_export['revenue']:,.0f}), so test targeted campaigns there first.",
 f"7. REDUCE REFUNDS: returns equal {refund_rate:.1%} of revenue. Check which products are returned most and fix "
   f"descriptions, packaging or quality.",
]
for r in recs:
    print(r, "\n")

1. PROTECT CHAMPIONS: 2,040 customers produce 77% of revenue. Launch a loyalty / trade-account programme with early access to new stock. 

2. WIN BACK AT-RISK CUSTOMERS: 886 previously frequent buyers have gone quiet (avg 276 days since last order) and represent £2,119,762 of past revenue. Send a targeted win-back offer. 

3. FIX RETENTION: only ~21% of new customers return the next month. Add a follow-up email/discount for the first 30 days after a first order. 

4. PLAN FOR PEAK SEASON: Sep-Nov brings 36% of the revenue in the data. Secure stock and marketing budget before September. 

5. REVIEW WEAK CATEGORY: 'Cards, Paper & Gift Wrap' earns the least per product (£3,330). Cut slow SKUs or bundle them with 'Bags & Storage' best-sellers. 

6. GROW EXPORTS: the UK is 85% of revenue. Ireland is the largest export market (£633,269), so test targeted campaigns there first. 

7. REDUCE REFUNDS: returns equal 7.4% of revenue. Check which products are returned most and fix descriptions, pac

## 12. Conclusion

The business earned **£19.7M from 39,526 orders and 5,853 identified customers** over two years. Growth depends on a small group of loyal trade customers and on the Christmas season.

**Key findings**
1. **Revenue is concentrated:** Champions (35% of customers) bring in 77% of revenue, and the top 10% of customers bring in 64%.
2. **Strong seasonality:** September–November delivers 36% of revenue, peaking in November.
3. **Retention is weak:** only 21% of new customers return the next month, and 886 valuable customers have gone inactive (£2.1M in past revenue).
4. **Product mix:** Bags & Storage performs best per product, while Cards, Paper & Gift Wrap is the weakest named category.
5. **Market dependence:** the UK provides 85.5% of revenue; Ireland and the Netherlands are the strongest export markets.
6. **Returns:** refunds equal 7.4% of revenue.

**Recommended actions**
1. Launch a loyalty programme for Champions and a win-back campaign for At-Risk customers.
2. Secure stock and marketing budget before September, and add a 30-day follow-up offer for new customers.
3. Grow exports in Ireland and the Netherlands, review the Cards, Paper & Gift Wrap range, and investigate the most returned products.

**Interactive dashboard:** submitted separately as a Streamlit app (`app.py`).

**Limitations:** no cost data (so no margin analysis); categories are keyword-based approximations; about 23% of order lines have no customer ID and are excluded from customer-level metrics.